# Location Tracker Server

Run this notebook locally to start the backend API server and connect to your local MySQL database. 

**Pre-requisite:** Open MySQL Workbench, open the `schema.sql` file from the `database` folder, and run it to create the database and table.

In [ ]:
!pip install fastapi uvicorn mysql-connector-python pydantic nest-asyncio

### Configure the Database Credentials below
Change `user` and `password` to match your local MySQL Server settings. Then run the cell to start the server.

In [ ]:
import mysql.connector
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import nest_asyncio
import uvicorn

# nest_asyncio allows uvicorn/fastapi to run inside Jupyter Notebooks
nest_asyncio.apply()

app = FastAPI(title="Location Tracker API")

# Enable CORS for frontend so your HTML page can access this API
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# -----------------------------------------
# UPDATE THESE WITH YOUR MYSQL CREDENTIALS
# -----------------------------------------
DB_CONFIG = {
    'host': '127.0.0.1',
    'user': 'root',          # MySQL username
    'password': 'password',  # MySQL password
    'database': 'location_db'
}

def get_db_connection():
    try:
        connection = mysql.connector.connect(**DB_CONFIG)
        return connection
    except mysql.connector.Error as err:
        print(f"Database connection failed: {err}")
        return None

class LocationData(BaseModel):
    latitude: float
    longitude: float

@app.post("/api/track")
async def track_location(data: LocationData):
    conn = get_db_connection()
    if not conn:
        raise HTTPException(status_code=500, detail="Database connection failed")
    
    cursor = conn.cursor()
    query = "INSERT INTO locations (latitude, longitude) VALUES (%s, %s)"
    values = (data.latitude, data.longitude)
    
    try:
        cursor.execute(query, values)
        conn.commit()
        return {"status": "success", "message": "Location logged successfully!"}
    except Exception as e:
        conn.rollback()
        raise HTTPException(status_code=500, detail=str(e))
    finally:
        cursor.close()
        conn.close()

@app.get("/api/locations")
async def get_locations():
    conn = get_db_connection()
    if not conn:
        raise HTTPException(status_code=500, detail="Database connection failed")
        
    cursor = conn.cursor(dictionary=True)
    # Get the latest 100 tracked locations
    query = "SELECT id, latitude, longitude, timestamp FROM locations ORDER BY timestamp DESC LIMIT 100"
    cursor.execute(query)
    rows = cursor.fetchall()
    cursor.close()
    conn.close()
    return rows

print("----------------------------------------------------------")
print("Starting FastAPI server on http://127.0.0.1:8000")
print("You can now open the frontend index.html in your browser!")
print("----------------------------------------------------------\n")

uvicorn.run(app, host="127.0.0.1", port=8000)
